# Politicians Network — Data Collection
Collects politicians active during **1900–1950** from Wikipedia/Wikidata.

### Pipeline
1. Fetch top Wikipedia pageviews for 2024 (parallelised)
2. Batch-resolve Wikidata IDs (50 titles per request)
3. Filter politicians via SPARQL (bulk, not one-by-one)
4. Enrich with position, nationality, birth/death info (paginated)

In [1]:
import requests
import time
import json
import pandas as pd
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timedelta
from tqdm import tqdm

In [2]:
# ── Shared session & constants ────────────────────────────────────────────────
S = requests.Session()
S.headers.update({"User-Agent": "PoliticiansNetwork/1.0 (kerem.ozemre@icloud.com)"})

BASE_PAGEVIEWS = "https://wikimedia.org/api/rest_v1/metrics/pageviews/top/en.wikipedia/all-access"
BASE_WIKI      = "https://en.wikipedia.org/w/api.php"
SPARQL_URL     = "https://query.wikidata.org/sparql"

SKIP_PREFIXES = ("Special:", "Main_Page", "Wikipedia:", "Help:", "File:", "Portal:")


def get_with_retry(url, params=None, session=S, max_retries=4):
    """GET with exponential backoff on 429/5xx."""
    for attempt in range(max_retries):
        try:
            r = session.get(url, params=params, timeout=20)
            if r.status_code == 429:
                wait = 2 ** attempt
                print(f"  Rate-limited, waiting {wait}s…")
                time.sleep(wait)
                continue
            r.raise_for_status()
            return r
        except requests.RequestException as e:
            if attempt == max_retries - 1:
                raise
            time.sleep(2 ** attempt)
    raise RuntimeError(f"Failed after {max_retries} retries: {url}")

## 1 — Fetch top Wikipedia pageviews (parallelised)

In [3]:
def fetch_day_views(date_str):
    """Return list of article dicts for one day (YYYY/MM/DD)."""
    url = f"{BASE_PAGEVIEWS}/{date_str}"
    try:
        r = get_with_retry(url)
        return r.json()["items"][0]["articles"]
    except Exception as e:
        print(f"  Error on {date_str}: {e}")
        return []


def get_top_pages_2024(n_days=360, max_workers=8):
    """Fetch top-1000 articles per day for n_days in 2024, parallelised."""
    start  = datetime(2024, 1, 1)
    days   = [(start + timedelta(days=i)).strftime("%Y/%m/%d") for i in range(n_days)]
    counts = defaultdict(int)

    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        futures = {pool.submit(fetch_day_views, d): d for d in days}
        for future in tqdm(as_completed(futures), total=len(days), desc="Fetching pageviews"):
            for article in future.result():
                title = article["article"]
                if not title.startswith(SKIP_PREFIXES):
                    counts[title] += article["views"]

    sorted_pages = sorted(counts.items(), key=lambda x: x[1], reverse=True)
    return [title for title, _ in sorted_pages]


page_views_ranked = get_top_pages_2024(n_days=360)
print(f"Unique pages collected: {len(page_views_ranked)}")
print("Top 20:", page_views_ranked[:20])

Fetching pageviews: 100%|██████████| 360/360 [00:02<00:00, 125.39it/s]

Unique pages collected: 40460
Top 20: ['Cleopatra', 'Deaths_in_2024', 'YouTube', 'XXXTentacion', 'Pornhub', '2024_United_States_presidential_election', 'Kamala_Harris', 'Donald_Trump', '.xxx', 'Indian_Premier_League', 'Deadpool_&_Wolverine', 'Project_2025', 'ChatGPT', 'Taylor_Swift', 'Elon_Musk', '2024_Indian_general_election', '2020_United_States_presidential_election', 'United_States', 'Facebook', 'XXX_(2002_film)']


## 2 — Batch-resolve Wikidata IDs (50 titles per request)

In [4]:
def get_wikidata_ids_batch(titles):
    """
    Resolve up to 50 Wikipedia titles → Wikidata QIDs in one API call.
    Returns dict {title: qid_or_None}.
    """
    params = {
        "action":  "query",
        "titles":  "|".join(titles),
        "prop":    "pageprops",
        "ppprop":  "wikibase_item",
        "format":  "json",
    }
    r = get_with_retry(BASE_WIKI, params=params)
    pages = r.json()["query"]["pages"]

    # Handle redirects: normalised titles come back under "normalized"
    normalised = {n["from"]: n["to"] for n in r.json()["query"].get("normalized", [])}

    result = {}
    for page in pages.values():
        qid   = page.get("pageprops", {}).get("wikibase_item")
        ptitle = page.get("title", "")
        result[ptitle] = qid

    # Map original (pre-normalisation) titles back
    for orig, norm in normalised.items():
        if norm in result:
            result[orig] = result[norm]
    return result


def resolve_all_qids(titles, batch_size=50):
    """Resolve all titles to QIDs using batched requests."""
    title_to_qid = {}
    batches = [titles[i:i+batch_size] for i in range(0, len(titles), batch_size)]
    for batch in tqdm(batches, desc="Resolving Wikidata IDs"):
        title_to_qid.update(get_wikidata_ids_batch(batch))
        time.sleep(0.05)   # be polite; much faster than 1 req/title
    return title_to_qid


# Use top 5000 candidates to keep runtime reasonable
candidates = page_views_ranked[:5000]
title_to_qid = resolve_all_qids(candidates)
qids = [qid for qid in title_to_qid.values() if qid]
print(f"Resolved {len(qids)} QIDs from {len(candidates)} titles")

Resolving Wikidata IDs: 100%|██████████| 100/100 [00:20<00:00,  4.78it/s]

Resolved 9229 QIDs from 5000 titles


## 3 — Filter to politicians via SPARQL (bulk)

In [5]:
def run_sparql(query, retries=4):
    """Execute a SPARQL query against Wikidata with retry logic."""
    for attempt in range(retries):
        try:
            r = S.get(
                SPARQL_URL,
                params={"query": query},
                headers={"Accept": "application/json"},
                timeout=60,
            )
            if r.status_code == 429:
                time.sleep(2 ** attempt)
                continue
            r.raise_for_status()
            return r.json()
        except Exception as e:
            if attempt == retries - 1:
                raise
            time.sleep(2 ** attempt)
    raise RuntimeError("SPARQL query failed after retries")


def filter_politicians_sparql(qids, batch_size=200):
    """
    Given a list of QIDs, return only those that are politicians (P106=Q82955).
    Batched to avoid SPARQL query-length limits.
    """
    politician_qids = set()
    batches = [qids[i:i+batch_size] for i in range(0, len(qids), batch_size)]

    for batch in tqdm(batches, desc="Filtering politicians"):
        values = " ".join(f"wd:{q}" for q in batch)
        query  = f"""
        SELECT ?person WHERE {{
          VALUES ?person {{ {values} }}
          ?person wdt:P106 wd:Q82955 .
        }}
        """
        data = run_sparql(query)
        for item in data["results"]["bindings"]:
            politician_qids.add(item["person"]["value"].split("/")[-1])
        time.sleep(0.5)

    return politician_qids


politician_qids = filter_politicians_sparql(qids)
print(f"Found {len(politician_qids)} politicians among candidates")

Filtering politicians: 100%|██████████| 47/47 [00:34<00:00,  1.35it/s]

Found 297 politicians among candidates


## 4 — Enrich: positions, nationality, birth/death (paginated, 1900–1950 filter)

In [ ]:
import time
import re
from requests.exceptions import ReadTimeout

def run_sparql_with_retry(query, retries=5):
    for attempt in range(retries):
        try:
            return run_sparql(query)
        except ReadTimeout:
            print(f"Timeout, retrying ({attempt+1}/{retries})...")
            time.sleep(2 * (attempt + 1))
    raise Exception("Max retries exceeded")
def make_query(after_uri=None, page_size=300):
    cursor_filter = f'FILTER(?person > <{after_uri}>)' if after_uri else ''
    return f"""
    #TOOL: politician-fetcher
    #TIMEOUT: 60000

    SELECT
      ?person
      ?personLabel
      ?positionLabel
      ?start
      ?end
      ?nationalityLabel
      ?birth
      ?death
    WHERE {{
      ?person wdt:P106 wd:Q82955 .
      ?person p:P39 ?statement .
      ?statement ps:P39 ?position .

      OPTIONAL {{ ?statement pq:P580 ?start. }}
      OPTIONAL {{ ?statement pq:P582 ?end. }}
      OPTIONAL {{ ?person wdt:P27 ?nat. }}
      OPTIONAL {{ ?person wdt:P569 ?birth. }}
      OPTIONAL {{ ?person wdt:P570 ?death. }}

      FILTER(
        (!BOUND(?end)   || ?end   >= "1900-01-01T00:00:00Z"^^xsd:dateTime) &&
        (!BOUND(?start) || ?start <= "1950-12-31T23:59:59Z"^^xsd:dateTime)
      )

      {cursor_filter}

      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }}
    ORDER BY ?person
    LIMIT {page_size}
    """
def get_politicians_fast(page_size=300):
    all_rows = []
    after_uri = None
    page_num = 1

    while True:
        print(f"Fetching page {page_num}...")

        data = run_sparql_with_retry(make_query(after_uri, page_size))
        bindings = data["results"]["bindings"]

        if not bindings:
            break

        for item in bindings:
            all_rows.append({
                "wikidata": item["person"]["value"],
                "name": item.get("personLabel", {}).get("value"),
                "position": item.get("positionLabel", {}).get("value"),
                "start": parse_wikidata_date(item.get("start", {}).get("value")),
                "end": parse_wikidata_date(item.get("end", {}).get("value")),
                "nationality": item.get("nationalityLabel", {}).get("value"),
                "birth_date": parse_wikidata_date(item.get("birth", {}).get("value")),
                "death_date": parse_wikidata_date(item.get("death", {}).get("value")),
            })

        print(f"  -> {len(bindings)} rows")

        if len(bindings) < page_size:
            break

        after_uri = bindings[-1]["person"]["value"]
        page_num += 1
        time.sleep(2)

    return deduplicate_people(all_rows)
from collections import defaultdict

def deduplicate_people(rows):
    people = {}

    for r in rows:
        pid = r["wikidata"]

        if pid not in people:
            people[pid] = {
                "wikidata": pid,
                "name": r["name"],
                "positions": set(),
                "nationality": r["nationality"],
                "birth_date": r["birth_date"],
                "death_date": r["death_date"],
                "start": r["start"],
                "end": r["end"],
            }

        if r["position"]:
            people[pid]["positions"].add(r["position"])

    # convert sets to lists
    for p in people.values():
        p["positions"] = list(p["positions"])

    return list(people.values())

raw_rows = get_politicians_fast(page_size=300)
print(raw_rows[:3])



Fetching page 1...


## 5 — Build final DataFrame & deduplicate

In [ ]:
df = pd.DataFrame(raw_rows)

# Dates are already clean 'YYYY-MM-DD' strings from parse_wikidata_date.
# Cast to proper date type for sorting/filtering.
for col in ["start", "end", "birth_date", "death_date"]:
    df[col] = pd.to_datetime(df[col], format="%Y-%m-%d", errors="coerce").dt.date

# Drop rows with no name at all (no English label on Wikidata)
df = df.dropna(subset=["name"])

# Deduplicate: one row per (person, position) — keep earliest start
df = (
    df.sort_values("start", na_position="last")
      .drop_duplicates(subset=["wikidata", "position"])
)

# Cross-reference with pageview politicians
qid_col = df["wikidata"].str.extract(r"(Q\d+)")[0]
df["in_pageview_top"] = qid_col.isin(politician_qids)

df = df.reset_index(drop=True)
print(f"Final dataset: {len(df)} rows, {df['wikidata'].nunique()} unique politicians")
df.head(10)


## 6 — Save results

In [ ]:
df.to_csv("politicians_1900_1950.csv", index=False)
print("Saved -> politicians_1900_1950.csv")

# Quick summary
print(f"\nPoliticians with a name:      {df['name'].notna().sum()}")
print(f"Politicians with nationality: {df['nationality'].notna().sum()}")
print(f"Politicians with birth date:  {df['birth_date'].notna().sum()}")

print("\n-- Nationality breakdown (top 20) --")
print(df["nationality"].value_counts().head(20).to_string())

print("\n-- Sample rows --")
print(df[["name", "position", "start", "end", "nationality", "birth_date"]].head(20).to_string(index=False))
